### Importing packages

In [118]:
import pandas as pd                                                                     # type: ignore

### Importing dataset

In [119]:
test_data_X = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/test-data-Dakar-map-features.csv")
test_data_y_lt1 = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/test-data-Dakar-map-target-lt1.csv", index_col=False)

train_data_X = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/train-data-Dakar-map-features.csv", index_col=False)
train_data_y_lt1 = pd.read_csv("/home/mendrika/mendrika-phd/codes/nflics/model-ML-Dakar/map/train-data-Dakar-map-target-lt1.csv", index_col=False)

### Choosing lead time

In [120]:
lead_time = 6

### Choosing data split

In [121]:
dataset = "test"
if dataset == "train":
    data = train_data_X
    target = train_data_y_lt1
else:
    data = test_data_X
    target = test_data_y_lt1

In [122]:
original_data = data.copy()

### Finding exact rows after given time (hh, mm)

In [123]:
data['datetime'] = pd.to_datetime(data[['year', 'month', 'day', 'hour', 'minute']])
data = data.sort_values(by='datetime').reset_index(drop=True)

In [124]:
def find_exact_row_after_given_hours(row, hours, minutes, df):
    target_time = row['datetime'] + pd.Timedelta(hours=hours, minutes=minutes)
    corresponding_row = df[df['datetime'] == target_time]
    if not corresponding_row.empty:
        return corresponding_row.index[0]  # Return the index of the corresponding row
    else:
        return None

In [125]:
data['row_index_X0_30'] = data.apply(find_exact_row_after_given_hours, args=(0, 30, data), axis=1)
data['row_index_X0_60'] = data.apply(find_exact_row_after_given_hours, args=(1, 0, data), axis=1)
data['row_index_X0_90'] = data.apply(find_exact_row_after_given_hours, args=(1, 30, data), axis=1)
data['row_index_X0_120'] = data.apply(find_exact_row_after_given_hours, args=(2, 0, data), axis=1)

One hour lead time

This is relative to 120 as it's the nowcast origin

+1 as $t_0$ is at 120 minutes

### Find the target index for the chosen lead time for each row

We need $t_0$, $t_1$ and $t_{lt}$

This it to get the corresponding target value after one hour

In [126]:
data['row_index_Cb'] = data.apply(find_exact_row_after_given_hours, args=(lead_time + 1 , 0, data), axis=1)      # since the target is at t0+1 h

In [127]:
columns_to_keep = original_data.keys().to_list()

In [128]:
# Function to combine current row with rows based on indices, retaining original column order
def combine_current_and_rows(row, data):
    # Extract the row indices from the current row
    indices = [row['row_index_X0_30'], row['row_index_X0_60'], row['row_index_X0_90'], row['row_index_X0_120']]
    
    # Fetch the current row (filter only relevant columns)
    current_row = row[columns_to_keep].copy()
    
    # List to store the rows
    rows_to_combine = [current_row]
    
    # Fetch rows corresponding to the indices, rename columns with suffix to avoid duplicates
    for i, idx in enumerate(indices):
        if pd.notna(idx):
            # Fetch the row, keep only relevant columns, and rename them with suffix
            fetched_row = data.loc[idx, columns_to_keep].rename(lambda col: f"{col}_{(i+1)*30}")
            rows_to_combine.append(fetched_row)
        else:
            # If the index is NaN, create an empty Series with the same columns as the current row
            empty_row = pd.Series(index=[f"{col}_{(i+1)*30}" for col in columns_to_keep])
            rows_to_combine.append(empty_row)
    
    # Concatenate the current row with the fetched rows side by side (maintain column order)
    combined_row = pd.concat(rows_to_combine, axis=0)
    
    return combined_row
# Apply the function row by row to get combined data
combined_data = data.apply(combine_current_and_rows, args=(data,), axis=1)

# Convert the combined series into a DataFrame while keeping the original index
combined_df = pd.DataFrame(combined_data, index=data.index)

### Taking target values

In [129]:
combined_df = pd.DataFrame(combined_data, index=data.index)

In [130]:
combined_df = combined_df.dropna()

In [131]:
combined_df

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size1_120,size2_120,size3_120,size4_120,size5_120,ds1_120,ds2_120,ds3_120,ds4_120,ds5_120
0,2020,6,1,0,0,11.32,-16.08,10.11,-15.77,11.37,...,5925.0,1675.0,0.0,0.0,0.0,139.09,155.36,400.00,400.00,400.0
2,2020,6,1,0,30,10.20,-17.07,10.06,-14.11,10.24,...,3775.0,975.0,0.0,0.0,0.0,132.24,161.53,400.00,400.00,400.0
10,2020,6,1,16,30,11.46,-12.76,11.64,-13.25,14.00,...,950.0,15575.0,2575.0,1300.0,0.0,133.28,135.95,138.81,164.54,400.0
11,2020,6,1,16,45,11.50,-12.76,11.64,-13.30,14.00,...,7600.0,1100.0,18800.0,1375.0,0.0,131.10,134.62,141.11,161.62,400.0
12,2020,6,1,17,0,12.13,-12.53,11.68,-13.30,14.00,...,8775.0,20450.0,625.0,0.0,0.0,130.27,139.18,159.55,400.00,400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17825,2022,9,30,19,45,11.50,-10.51,11.59,-11.82,11.73,...,5550.0,5425.0,0.0,0.0,0.0,158.40,181.57,400.00,400.00,400.0
17826,2022,9,30,20,0,11.28,-12.44,11.59,-11.82,11.77,...,4000.0,3250.0,1025.0,0.0,0.0,159.11,179.79,211.92,400.00,400.0
17827,2022,9,30,20,15,11.46,-10.56,16.35,-12.18,11.64,...,550.0,1900.0,625.0,2375.0,0.0,156.59,161.05,180.25,211.92,400.0
17828,2022,9,30,20,30,11.41,-10.56,11.64,-12.98,16.22,...,375.0,225.0,1575.0,1250.0,0.0,154.85,159.35,161.76,211.08,400.0


In [132]:
o = data['row_index_Cb'][combined_df.index].dropna()

In [133]:
combined_df.loc[o.index]

,year,month,day,hour,minute,lat1,lon1,lat2,lon2,lat3,...,size1_120,size2_120,size3_120,size4_120,size5_120,ds1_120,ds2_120,ds3_120,ds4_120,ds5_120
41,2020,6,2,10,45,11.86,-10.38,11.23,-11.55,14.00,...,5100.0,17975.0,0.0,0.0,0.0,169.52,172.54,400.00,400.00,400.00
42,2020,6,2,11,0,11.86,-10.47,10.92,-12.04,11.28,...,6150.0,24075.0,0.0,0.0,0.0,168.19,172.74,400.00,400.00,400.00
43,2020,6,2,11,15,12.71,-10.11,12.00,-10.56,10.96,...,30450.0,5975.0,12100.0,0.0,0.0,153.39,164.62,179.49,400.00,400.00
44,2020,6,2,11,30,12.08,-10.65,11.37,-11.86,11.01,...,7050.0,2300.0,5475.0,4525.0,2525.0,150.00,156.24,161.48,162.11,175.42
45,2020,6,2,11,45,12.13,-10.74,11.41,-11.95,11.05,...,8100.0,2500.0,5275.0,1550.0,2350.0,148.69,151.16,158.82,159.51,171.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17783,2022,9,29,20,30,15.81,-11.14,12.04,-16.94,12.71,...,20450.0,500.0,850.0,9350.0,600.0,50.49,68.24,75.17,119.50,145.84
17784,2022,9,29,20,45,15.86,-11.19,12.85,-16.71,12.08,...,22675.0,8175.0,1725.0,2250.0,17175.0,49.73,118.14,156.03,172.86,178.84
17785,2022,9,29,21,0,15.86,-11.23,10.33,-11.28,12.94,...,22800.0,5175.0,2050.0,2075.0,15900.0,48.51,116.73,154.46,176.47,178.57
17786,2022,9,29,21,15,11.91,-13.66,12.98,-16.40,11.28,...,19375.0,1500.0,3975.0,13850.0,1325.0,46.32,59.14,114.02,147.09,153.67


In [134]:
target

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17833,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17834,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17835,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17836,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [135]:
target_index = data['row_index_Cb'][combined_df.index].dropna()
target = target.loc[target_index]
combined_df = combined_df.loc[target_index.index]

### Writing data to disk

In [136]:
combined_df.to_csv(f"./2h-before/{dataset}-Dakar-map-features-many-timesteps-30min-lt{lead_time}-2h-before.csv", index=False)
target.to_csv(f"./2h-before/{dataset}-Dakar-map-target-many-timesteps-30min-lt{lead_time}-2h-before.csv", index=False)